# 01 — EDA: Quantitative Distribution Audit

Each section is a **checkpoint**: run the cell, evaluate the output, then proceed.
All logic lives in `src/` — this notebook only calls it and displays results.

```
STEP 1  Load raw CSVs
STEP 2  Join by study (uid)
STEP 3  Integrity check + sample findings text
STEP 4  CheXbert labels  ← slow on CPU (~10–20 min); cached after first run
STEP 5  Label prevalences
STEP 6  K_eff and entropy
STEP 7  Class ESS
STEP 8  Tail mass
STEP 9  Co-occurrence matrix
STEP 10 Save figures
```

In [ ]:
import logging
import yaml
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.dpi'] = 120
logging.basicConfig(level=logging.INFO, format='%(levelname)s  %(message)s')

with open('../params.yaml') as f:
    params = yaml.safe_load(f)

RAW_DIR       = Path('../') / params['data']['raw_dir']
PROCESSED_DIR = Path('../') / params['data']['processed_dir']
FIGURES_DIR   = Path('../reports/figures')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('RAW_DIR      :', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)
print('Files in raw :', list(RAW_DIR.glob('*')) if RAW_DIR.exists() else 'MISSING')

## STEP 1 — Load raw CSVs
Evaluate: correct column names? expected row counts?

In [ ]:
from src.data.load import load_reports, load_projections

reports     = load_reports(RAW_DIR)
projections = load_projections(RAW_DIR)

print(f'reports:     {reports.shape}     columns: {reports.columns.tolist()}')
print(f'projections: {projections.shape}  columns: {projections.columns.tolist()}')

In [ ]:
# Peek at both tables
print('=== reports (first 3 rows) ===')
display(reports.head(3))
print('\n=== projections (first 6 rows) ===')
display(projections.head(6))

In [ ]:
# Null counts per column — know what's missing before joining
print('=== reports null counts ===')
print(reports.isnull().sum().to_string())
print('\n=== projections null counts ===')
print(projections.isnull().sum().to_string())

## STEP 2 — Join by study (uid)
Evaluate: one row per study? frontal + lateral images collapsed to lists?

In [ ]:
from src.data.load import build_study_df

df = build_study_df(reports, projections)

print(f'Shape after join: {df.shape}  (should be ≈ n_unique_uids x n_cols)')
print(f'Unique UIDs in reports:     {reports["uid"].nunique()}')
print(f'Unique UIDs after join:     {df["uid"].nunique()}')
print(f'Columns: {df.columns.tolist()}')
display(df.head(3))

In [ ]:
# Check how many studies have frontal and/or lateral images
if 'frontal' in df.columns:
    n_frontal  = df['frontal'].apply(lambda x: len(x) > 0 if isinstance(x, list) else False).sum()
    print(f'Studies with frontal image:  {n_frontal} / {len(df)}')
if 'lateral' in df.columns:
    n_lateral  = df['lateral'].apply(lambda x: len(x) > 0 if isinstance(x, list) else False).sum()
    print(f'Studies with lateral image:  {n_lateral} / {len(df)}')

# Distribution of projection counts per study
proj_counts = projections.groupby('uid')['filename'].count()
proj_counts.value_counts().sort_index().rename('n_studies').to_frame()

## STEP 3 — Integrity check + sample findings text
Evaluate: how many studies are unusable (no findings)? Does the text look clinical?

In [ ]:
from src.data.load import validate_integrity

stats = validate_integrity(df)
pd.Series(stats).rename('count').to_frame()

In [ ]:
# Drop studies with no findings text (can't be used as generation targets)
mask = df['findings'].notna() & (df['findings'].astype(str).str.strip() != '')
df_clean = df[mask].reset_index(drop=True)

print(f'Before: {len(df)} studies')
print(f'After dropping empty findings: {len(df_clean)} studies')
print(f'Dropped: {len(df) - len(df_clean)}')

In [ ]:
# Read 3 example studies — check text quality and clinical relevance
for i in [0, 1, 2]:
    row = df_clean.iloc[i]
    print(f"\n{'='*60}")
    print(f"UID: {row['uid']}")
    print(f"Indication : {str(row.get('indication', 'N/A'))[:150]}")
    print(f"Findings   : {str(row['findings'])[:300]}")
    print(f"Impression : {str(row.get('impression', 'N/A'))[:150]}")

## STEP 4 — CheXbert labels

**First run:** downloads ~400 MB CheXbert model from HuggingFace, then runs BERT
over all ~3 k reports. Takes **~10–20 min on CPU**. Results are cached to disk.

**Subsequent runs:** loads instantly from the cache parquet.

Evaluate: does the label matrix look sensible? Are the all-zero rows (no finding) dominant?

In [ ]:
from src.data.labels import label_dataframe, CHEXBERT_LABELS

LABEL_CACHE = PROCESSED_DIR / 'chexbert_labels.parquet'

print(f'Cache exists: {LABEL_CACHE.exists()}')
print('Running label_dataframe (loading from cache if available)...')

df_labeled = label_dataframe(
    df_clean,
    uncertain_policy=params['labels']['uncertain_policy'],
    device=params['labels']['device'],
    cache_path=LABEL_CACHE,
)

print(f'\nShape: {df_labeled.shape}')
print(f'Label columns added: {CHEXBERT_LABELS}')

In [ ]:
# Inspect label matrix — first 10 rows
display(df_labeled[['uid', 'findings'] + CHEXBERT_LABELS].head(10))

In [ ]:
# Per-label positive count
label_counts = df_labeled[CHEXBERT_LABELS].sum().sort_values(ascending=False)
print('Positive studies per label:')
print(label_counts.to_string())

In [ ]:
# Save the labeled dataset so terminal scripts can also use it
out = PROCESSED_DIR / 'dataset_labeled.parquet'
df_labeled.to_parquet(out, index=False)
print(f'Saved to {out}')

## STEP 5 — Label prevalences
Evaluate: how dominated is this dataset by "No Finding"? Which labels are rare?

In [ ]:
from src.eda.distribution_audit import compute_prevalences

label_matrix = df_labeled[CHEXBERT_LABELS].to_numpy(dtype=np.float32)
prevalences  = compute_prevalences(label_matrix)

print('Label prevalences (fraction of studies):')
print(prevalences.sort_values(ascending=False).to_string())

In [ ]:
sorted_prev = prevalences.sort_values(ascending=False)
colors = ['#d62728' if v < 0.05 else '#1f77b4' for v in sorted_prev.values]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(sorted_prev)), sorted_prev.values, color=colors)
ax.set_xticks(range(len(sorted_prev)))
ax.set_xticklabels(sorted_prev.index, rotation=45, ha='right', fontsize=9)
ax.axhline(0.05, color='red', linestyle='--', linewidth=0.8, label='rare threshold (5%)')
ax.set_ylabel('Prevalence')
ax.set_title('CheXbert Label Prevalences — IU CXR  (red = rare, p < 5%)')
ax.legend()
plt.tight_layout()
plt.show()

## STEP 6 — K_eff and Shannon entropy
Evaluate: how concentrated is the dataset?
K_eff = 1 → one label dominates entirely. K_eff = 14 → all equal.

In [ ]:
from src.eda.distribution_audit import compute_keff

keff = compute_keff(prevalences)
p = prevalences.values.clip(1e-9)
p_norm = p / p.sum()
entropy = float(-np.sum(p_norm * np.log(p_norm)))

print(f'Shannon entropy H  = {entropy:.3f} nats')
print(f'K_eff = exp(H)      = {keff:.2f}  (out of 14 possible)')
print()
print('Interpretation: the dataset behaves as if there are only')
print(f'{keff:.1f} equally-common labels — heavily skewed toward "No Finding".')

## STEP 7 — Per-class ESS
Evaluate: if we up-weight to uniform prevalence, how many *effective* samples does each rare class have?
Low ESS = we don't have much real signal for that class even with reweighting.

In [ ]:
from src.eda.distribution_audit import compute_class_ess

class_ess = compute_class_ess(label_matrix, prevalences)
sorted_ess = class_ess.sort_values(ascending=True)

print('Per-class ESS (effective sample size ≈ n_positive):')
print(sorted_ess.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(range(len(sorted_ess)), sorted_ess.values, color='#2ca02c')
ax.set_yticks(range(len(sorted_ess)))
ax.set_yticklabels(sorted_ess.index, fontsize=9)
ax.set_xlabel('ESS')
ax.set_title('Per-class ESS — data-starved classes have the lowest bars')
plt.tight_layout()
plt.show()

## STEP 8 — Tail mass
Evaluate: what fraction of studies only have rare labels (p < 5%)?
These studies get almost no gradient signal under standard training.

In [ ]:
from src.eda.distribution_audit import compute_tail_mass

tail_mass = compute_tail_mass(label_matrix)
print(f'Tail mass: {tail_mass:.3f}  ({tail_mass*100:.1f}% of studies)')
print()
print('These are the studies the importance-weighted sampler is most')
print('important for — they\'re the first to be lost under uniform sampling.')

## STEP 9 — Label co-occurrence
Evaluate: which labels tend to appear together? Are there clinically expected patterns
(e.g. Edema + Pleural Effusion, Consolidation + Pneumonia)?

In [ ]:
from src.eda.distribution_audit import compute_cooccurrence
import seaborn as sns

cooccurrence = compute_cooccurrence(label_matrix)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    cooccurrence, annot=True, fmt='.2f', cmap='Blues',
    vmin=0, vmax=1, ax=ax, linewidths=0.4, annot_kws={'size': 7},
)
ax.set_title('P(column label | row label positive)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Top 5 co-occurring pairs (excluding diagonal)
cooc_flat = cooccurrence.copy()
np.fill_diagonal(cooc_flat.values, 0)
top = cooc_flat.unstack().sort_values(ascending=False).head(10)
print('Top 10 label pairs by conditional co-occurrence P(col | row):')
print(top.rename('P(col|row)').to_string())

## STEP 10 — Save figures to reports/figures/
These are the figures that go into the writeup.

In [ ]:
from src.eda.distribution_audit import _plot_prevalences, _plot_cooccurrence, _plot_class_ess

_plot_prevalences(prevalences, FIGURES_DIR / 'label_prevalences.png')
_plot_cooccurrence(cooccurrence, FIGURES_DIR / 'cooccurrence_matrix.png')
_plot_class_ess(class_ess, FIGURES_DIR / 'class_ess.png')

print('Saved figures:')
for f in sorted(FIGURES_DIR.glob('*.png')):
    print(' ', f)

## Summary

Fill in the numbers after running each cell:

| Metric | Value |
|--------|-------|
| Total studies (with findings) | ??? |
| K_eff | ??? / 14 |
| Tail mass | ???% |
| Most prevalent label | ??? (???%) |
| Most data-starved class (ESS) | ??? |

**Next step:** `notebooks/02_baseline_zero_shot.ipynb` (requires GPU)